In [2]:
import pandas as pd

# This is a fresh import. we have imported the file directly from the internet. No folder was downloaded in the process
# This is a better solution than troubleshooting downloads, pandas can read a CSV directly from a URL, no manual downloads needed at all.
# This is actually how real data science workflows work — data is pulled straight into the code.

df = pd.read_csv("https://raw.githubusercontent.com/owid/covid-19-data/master/public/data/owid-covid-data.csv")

print(df.columns)
print(df.head())

Index(['iso_code', 'continent', 'location', 'date', 'total_cases', 'new_cases',
       'new_cases_smoothed', 'total_deaths', 'new_deaths',
       'new_deaths_smoothed', 'total_cases_per_million',
       'new_cases_per_million', 'new_cases_smoothed_per_million',
       'total_deaths_per_million', 'new_deaths_per_million',
       'new_deaths_smoothed_per_million', 'reproduction_rate', 'icu_patients',
       'icu_patients_per_million', 'hosp_patients',
       'hosp_patients_per_million', 'weekly_icu_admissions',
       'weekly_icu_admissions_per_million', 'weekly_hosp_admissions',
       'weekly_hosp_admissions_per_million', 'total_tests', 'new_tests',
       'total_tests_per_thousand', 'new_tests_per_thousand',
       'new_tests_smoothed', 'new_tests_smoothed_per_thousand',
       'positive_rate', 'tests_per_case', 'tests_units', 'total_vaccinations',
       'people_vaccinated', 'people_fully_vaccinated', 'total_boosters',
       'new_vaccinations', 'new_vaccinations_smoothed',
       't

## Research question: Did the rate of COVID-19 vaccination rollout correlate with a reduction in death rates across countries, and does this pattern differ between high-income and low-income nations?

To begin off, we will choose to do 10 countries (since all 218 countries is complicated). We will select these countries using stratified random sampling—meaning we will split them into high-income and low-income groups first based on World Bank classifications, and then randomly pick from each group. Before we choose any country, we have to filter them and check if those countries have any missing nulls or data missing so we only use countries with complete data. We will begin with looking at the US as part of our data.

We will use 2 variables:

- people_vaccinated_per_hundred: This variable shows us a percentage of people who got vaccinated/dosed at least once, tracked day by day. This number climbs up from 0 when a vaccination is created and people start rolling into hospitals but never reaches 100% as not all (100%) people are vaccinated ever. This variable will tell us how fast and how far the vaccinations rollout went. This is the "cause" side of our question, but it doesn't automatically mean people were saved yet; that is what we are testing.

  -> "How fast": Since this is tracked day by day, if a country x goes (always begins at 0 as nobody is vaccinated when the disease just spreads): 0, 10, 30, 50, 70 (final) in 2 months, the speed is lightning. The country is very fast since they were able to vaccinate 70% of their population in under 2 months. A steep climb means a rapid rollout.

  -> "How far": The "final" highest value a country eventually reaches determines how far their vaccination rollout went. A higher value means a larger portion of their population was covered by the rollout.

[Note on the Timing Problem]: Different countries started their rollouts at completely different calendar times. Richer countries got them early in 2021, while some poorer countries didn't get them until 2022. Because of this relative difference in time intervals, we cannot use a fixed calendar window. Instead, we will define each country's start date by when their people_vaccinated_per_hundred crosses a specific threshold, and then track them by "weeks since rollout start" so they line up on the same x-axis. 

    - Rollout dates start on the first non-zero vaccination date. [LIMITATION: PT-3]

- new_deaths_smoothed_per_million: This variable tracks COVID deaths per million people. It is important to note that this column exists for the entire pandemic, starting roughly a year before any vaccine existed, so we have to isolate the specific period connected to each country's rollout. If vaccines are successful, we want to see if this drops after the rollout begins. The data is also smoothed out because the data, in raw form, would be messy and potentially inaccurate. For example: Sundays are holidays for most workers at a hospital who put in data about death rates. On Monday, they would essentially put Sunday's and Monday's death rates on the same day making it seem like Sunday had 0 deaths while Monday had a deadly amount of deaths. The data is made smooth by making values a 7-day-average effectively removing such reporting inaccuracies.

Known Limitations:
- We have to honestly state a confounding variable in our report that we cannot fully solve: richer countries likely had more built-up natural immunity from prior infection waves by the time they started vaccinating. This natural immunity would also lower death rates independent of the vaccines, and our dataset cannot fully separate this effect from the vaccination's effect.
- We also cannot isolate the fact that some people might have built in immunities early on making vaccines affect on the person neglible as the person is already able to fend themselves.
- We are going to assume that for each country, certain handful of people were selected and treated seperately early on for the treatment and testing of the vaccine. This might certainly skew data as some people might have got a vaccine way earlier than the general public and we cannot find those specific people. We acknowledge this and will keep them in the dataset.


In [3]:
# df['continent'].unique()    # this gives all unique values in column = continent
print(df['location'].unique())  # this gives all unique values in column = location (country wise)

# Since i dont know the exact String for which US is called, i will look for it

u_Countries = df[df['location'].str.startswith('U', na=False)]['location'].unique() # this has all countries beginning with U
print(u_Countries)

# This tells us the exact String name with upper and lower case. its called "United States"

df_US = df[df['location'] == 'United States'][['people_vaccinated_per_hundred', 'new_deaths_smoothed_per_million', 'date']] 
print(df_US.head())

# lets look for null values, lesser the better
print(df_US['people_vaccinated_per_hundred'].isnull().sum())    # 796 null values 
print(df_US['new_deaths_smoothed_per_million'].isnull().sum())  # 5 null values

# Lets compare the total null values with the total population of US people takne 
print()
print(df_US.shape) # size = 1674

# thats terrible news. 796/1674 ~ 47.6% of the values are null. thats terrible. Almost half the data is missing. We need to loop through
# all countries like this to find the perfect 10 candidates.

# ~~ Testing ~~
# I want to find the first date at which a 'null' wasn't produced. We have so many nulls because this is pre-rollout era. When vaccines weren't
# publicly available or made

print()
print(df_US['people_vaccinated_per_hundred'].first_valid_index())   # this gives the first index without a null value

# lets find the date using the ID number from original df we got 403794

print(df_US.loc[403794, 'date'])    # this gives us the date based on the ID number from df

<StringArray>
[        'Afghanistan',              'Africa',             'Albania',
             'Algeria',      'American Samoa',             'Andorra',
              'Angola',            'Anguilla', 'Antigua and Barbuda',
           'Argentina',
 ...
             'Vatican',           'Venezuela',             'Vietnam',
               'Wales',   'Wallis and Futuna',      'Western Sahara',
               'World',               'Yemen',              'Zambia',
            'Zimbabwe']
Length: 255, dtype: str
<StringArray>
[                       'Uganda',                       'Ukraine',
          'United Arab Emirates',                'United Kingdom',
                 'United States',  'United States Virgin Islands',
 'Upper-middle-income countries',                       'Uruguay',
                    'Uzbekistan']
Length: 9, dtype: str
        people_vaccinated_per_hundred  new_deaths_smoothed_per_million  \
403451                            NaN                              NaN   
403

In [4]:
# lets attempt at making this for-loop 

All_Countries = df['location'].unique()

for country in All_Countries:  # Looping through all countries
    countryInQuestion = df[df['location'] == country]   # Data frame with all categories of each country
    validID = countryInQuestion['people_vaccinated_per_hundred'].first_valid_index() # finds the valid ID (as i tested with USA before)

    if pd.isna(validID):  # if validID is null...
        startDate = "No valid date"
    else:
        startDate = countryInQuestion.loc[validID, 'date']  # This finds the date using the validID
    nullVar1 = countryInQuestion['people_vaccinated_per_hundred'].isnull().sum()    
    nullVar2 = countryInQuestion['new_deaths_smoothed_per_million'].isnull().sum()
    percentageLost = 100*(nullVar1/countryInQuestion.shape[0]) # this is the percentage of null values compared to the total. will help determine
    # which countries to choose and which to not. Higher percentageLost, higher amounts of null value and will probably not choose those for 
    # candidates

    print(country + " " + "People Vaccinated: " + str(nullVar1) + " New Deaths: " + str(nullVar2) + " " + "percentage Lost: " + str(percentageLost) + " Start Date: " + str(startDate))

Afghanistan People Vaccinated: 1527 New Deaths: 5 percentage Lost: 91.21863799283155 Start Date: 2021-02-22
Africa People Vaccinated: 587 New Deaths: 5 percentage Lost: 35.065710872162484 Start Date: 2021-01-09
Albania People Vaccinated: 1445 New Deaths: 5 percentage Lost: 86.32019115890084 Start Date: 2021-01-10
Algeria People Vaccinated: 1647 New Deaths: 5 percentage Lost: 98.38709677419355 Start Date: 2021-01-29
American Samoa People Vaccinated: 1674 New Deaths: 5 percentage Lost: 100.0 Start Date: No valid date
Andorra People Vaccinated: 1615 New Deaths: 5 percentage Lost: 96.47550776583034 Start Date: 2021-01-25
Angola People Vaccinated: 1569 New Deaths: 5 percentage Lost: 93.72759856630825 Start Date: 2021-03-01
Anguilla People Vaccinated: 1605 New Deaths: 5 percentage Lost: 95.87813620071685 Start Date: 2021-02-04
Antigua and Barbuda People Vaccinated: 1580 New Deaths: 5 percentage Lost: 94.38470728793308 Start Date: 2021-02-16
Argentina People Vaccinated: 553 New Deaths: 9 perc

In [5]:
# I noticed a problem. We have a lot of weird titles such as Africa and Asia in the first 20-25 outputs as seen above. These are not countries
# These are continents. We don't want that. I will find a way to remove them in this cell

print(df['iso_code'])
df[df['location'].isin(['Africa', 'Asia', 'World'])][['location', 'iso_code']].drop_duplicates() # Most weird titles begin with "OWID"
# In our big loop above, we found 3 weird titles, so we tried to see what their 'iso_code' is from which we found OWID_AFR...
# OWID is not present in countries, its present in these weird titles only so we will remove all "OWID_" iso_code rows from the original df

df_cleaned = df[~df['iso_code'].str.startswith("OWID_")] 
df_cleaned['location'].unique()

All_Countries = df_cleaned['location'].unique() # this is needed as now we have all titles with actual countries and not weird titles

0         AFG
1         AFG
2         AFG
3         AFG
4         AFG
         ... 
429430    ZWE
429431    ZWE
429432    ZWE
429433    ZWE
429434    ZWE
Name: iso_code, Length: 429435, dtype: str


In [ ]:
# We repeat the entire loop for the now updated All_countries with all valid titles of countries.
# We need to do this as all these variables (countryInQuestion, ValidID etc.) are set to the last country that was checked in the previous huge 
# for loop (2 cells back)
# Variables here can be defined inside loops and used outside unlike java variables

# Now I want to decide a suitable cutoff for percentage loss. for example, if we had a country with 90% of dataloss due to null values,
# we could fix it using interpolation but those are values predicted through that 10% of data that we actually have. its not real values,
# its predicted which greatly undermines our initial research question. But we also have to account for the fact that most countries 
# have a loss of 90% or more. To find a suitable cutoff, we will make a dictionary called "results" instead of printing out stuff like we did
# in our previous big for loop

results = []

for country in All_Countries:
        countryInQuestion = df_cleaned[df_cleaned['location'] == country]  # This code of line accounts for all 6 years [2020=2026] 
        validID = countryInQuestion['people_vaccinated_per_hundred'].first_valid_index() 
        # Beta (defined on the next cell)
        lastID = countryInQuestion['people_vaccinated_per_hundred'].last_valid_index()

        # Alpha (defined below)
        countryPostRollout = countryInQuestion.loc[validID: lastID]

        if pd.isna(validID):  
                startDate = "No valid date"
        else:
                startDate = countryInQuestion.loc[validID, 'date']  
        #nullVar1 = countryInQuestion['people_vaccinated_per_hundred'].isnull().sum()    
        nullVar1 = countryPostRollout['people_vaccinated_per_hundred'].isnull().sum()    
        #nullVar2 = countryInQuestion['new_deaths_smoothed_per_million'].isnull().sum()
        nullVar2 = countryPostRollout['new_deaths_smoothed_per_million'].isnull().sum()    
        percentageLost = 100*(nullVar1/countryPostRollout.shape[0]) 

        results.append({"country": country, "People_Vaccinated_per_hundred nulls": nullVar1, "new_deaths_smoothed_per_million":nullVar2, "loss of data due to null (%)": percentageLost, "Start Date": startDate})
        # this line helps us add all the information into a list of columns for each country. We will then convert into a dataframe 

        # Replaced with dictionary called "results"
        # print(country + " " + "People Vaccinated: " + str(nullVar1) + " New Deaths: " + str(nullVar2) + " " + "percentage Lost: " + str(percentageLost) + " Start Date: " + str(startDate))


# Prevent text from wrapping to a new line if columns are wide
pd.set_option('display.width', 1000)
# Show the full string content inside cells without clipping words
pd.set_option('display.max_colwidth', None)

results_df = pd.DataFrame(results) # converted into a dataframe

print(results_df.head())
print()
print(results_df.shape)

# Alpha: This is a name given to a problem that applies only in this cell. I found this problem all the way after writing all the code you
# see on top, that are commented (before implementing countryPostRllout). I found the issue that countryInQuestion, while a valid variable for 
# validID and startdate, was not the best representation of nullVar1, nullVar2 and percentageLost. This is because the data 
# (before implementing countryPostRollout) consisted of time before the vaccines were given (basically, 2020-2026). If you comment the 
# countryPostRollout variable for calculating nullVar1, nullVar2 and percentageLost, you would see that percentageLOst is extremely high. 
# Majority of the countries have over 80% data lost due to null variables. I made a new variable that has the timeline from
# each of the country's specific startDate to 2026. The observation we see is that there isn't a significant chnage. THe percentageLost are still
# pretty high up to 88% and 100% also. (before adding beta)

# Alpha + beta extension: After adding beta, our timeline changes to (first valid index: last valid index). This essentially is our best data
# we can produce for now.

          country  People_Vaccinated_per_hundred nulls  new_deaths_smoothed_per_million  loss of data due to null (%)     Start Date
0     Afghanistan                                  896                                0                     85.906040     2021-02-22
1         Albania                                  745                                0                     76.488706     2021-01-10
2         Algeria                                  557                                0                     95.376712     2021-01-29
3  American Samoa                                 1674                                5                    100.000000  No valid date
4         Andorra                                  914                                0                     93.936280     2021-01-25

(237, 5)


In [ ]:
# I want to test something, I want to see if some year's worth of data was "excluded" or "not reported" in this. 90% data missing is rather
# suspcious and cannot necessarily be worked with. I will choose a country(USA for now) and see the head (first few rows) and tail 
# (bottom few rows) and see how many nulls occur. 

# Lets build the data from scratch first

df_US_full = df_cleaned[df_cleaned['location'] == 'United States']
validID_US = df_US_full['people_vaccinated_per_hundred'].first_valid_index()
US_PostRollout = df_US_full.loc[validID_US:]

# head and tail
print(US_PostRollout[['date', 'people_vaccinated_per_hundred']].head())
print(US_PostRollout[['date', 'people_vaccinated_per_hundred']].tail())

# It seems that 2024 had mostly nulls for the last few days of reports. 

# Beta: this applies to the cell before this one only
# I have an idea, what if we trim the back as well like we trimmed the front with startDate. startDate excluded all the nulls that existed
# before the first data that was collected. Lets trim it so that the timeline consists of the startDate (first data value) to the end date 
# (the last date recorded with a non-null value). After this, if percentages are high. We realistically cannot do much. We will have to move on.
# This method will essentally exclude all the nulls that follow after the last data recorded.

              date  people_vaccinated_per_hundred
403794  2020-12-13                           0.01
403795  2020-12-14                           0.01
403796  2020-12-15                           0.03
403797  2020-12-16                           0.07
403798  2020-12-17                           0.15
              date  people_vaccinated_per_hundred
405120  2024-07-31                            NaN
405121  2024-08-01                            NaN
405122  2024-08-02                            NaN
405123  2024-08-03                            NaN
405124  2024-08-04                            NaN


In [ ]:
# Ok, now that we have results_df, I want to see how many countries have lower than 90/85/80/75 percent dataloss due to null values

countsForNinty = results_df[results_df['loss of data due to null (%)'] < 90].shape[0]
print(countsForNinty)

countsForEightyFive = results_df[results_df['loss of data due to null (%)'] < 85].shape[0]
print(countsForEightyFive)

countsForEighty = results_df[results_df['loss of data due to null (%)'] < 80].shape[0]
print(countsForEighty)

countsForSeventyFive = results_df[results_df['loss of data due to null (%)'] < 75].shape[0]
print(countsForSeventyFive)

# This is pretty valid, 92 countries for 75% is rather promising. We could go lower but first, let us attempt finding 5 low income
# countries (listed according to World Bank). 
# [NOTE:] Our classifcations of low and high income countries directly come from World bank. I did not attempt to use any variables from
# within the data to decide which are low and high income countries.

# List: Afghanistan, Burkina Faso, Burundi, Central African Republic, Chad, Democratic Republic of Congo, Eritrea, Ethiopia, Gambia, Guinea, 
# Guinea-Bissau, North Korea, Liberia, Madagascar, Malawi, Mali, Mozambique, Niger, Rwanda, Sierra Leone, Somalia, South Sudan, Sudan, Syria, 
# Togo, Uganda, Yemen

low_Incomes = ["Afghanistan", "Burkina Faso", "Burundi", "Central African Republic", "Chad", "Democratic Republic of Congo", 
               "Eritrea", "Ethiopia", "Gambia", "Guinea", "Guinea-Bissau", "North Korea", "Liberia", "Madagascar", "Malawi", "Mali", 
               "Mozambique", "Niger", "Rwanda", "Sierra Leone", "Somalia", "South Sudan", "Sudan", "Syria", "Togo", "Uganda", "Yemen"]

TestingCutOff = results_df[results_df['loss of data due to null (%)'] < 88] # this gives us a testing df with countries with lesser than 88% loss
matchingLowIncome = TestingCutOff[TestingCutOff['country'].isin(low_Incomes)] # this checks aganist low income countries
#print(matchingLowIncome)

# Since, we are testing 10 countries for our entire research, we would need 5 low income and 5 higher income. to get 5 low income,
# we would have to account for 88% of percentage loss AT LEAST. any lower would give us lesser countries. 75% gives us 2. Clearly not enough
# This also gives us a valid observation that lower income countries often had most of their data missing despite trimming before rollout and 
# after the last vald data date.

# We essentially have our low income countries. They are stored in matchingLowIncome. Lets find 5 valid high income countries. we have 
# 87 countries which is a lot. We will use a random variable (later) if the list is too long to choose 5 valid high income countries with
# no bias. 

high_Incomes =  [
    "Andorra", "Antigua and Barbuda", "Australia", "Austria", "Bahamas", 
    "Bahrain", "Barbados", "Belgium", "Brunei", "Bulgaria", "Canada", 
    "Chile", "China", "Costa Rica", "Croatia", "Cyprus", "Czechia", 
    "Denmark", "Estonia", "Finland", "France", "Germany", "Greece", 
    "Guyana", "Hungary", "Iceland", "Ireland", "Israel", "Italy", 
    "Japan", "Kuwait", "Latvia", "Liechtenstein", "Lithuania", 
    "Luxembourg", "Malta", "Monaco", "Nauru", "Netherlands", 
    "New Zealand", "Norway", "Oman", "Palau", "Panama", "Poland", 
    "Portugal", "Qatar", "Romania", "Russia", "Saint Kitts and Nevis", 
    "San Marino", "Saudi Arabia", "Seychelles", "Singapore", "Slovakia", 
    "Slovenia", "South Korea", "Spain", "Sweden", "Switzerland", 
    "Trinidad and Tobago", "United Arab Emirates", "United Kingdom", 
    "United States", "Uruguay", "Hong Kong", "Taiwan"
]
print()
print()
matchingHighIncome = TestingCutOff[TestingCutOff['country'].isin(high_Incomes)] # Testing high income countries which are in the cutoff for 88% 
#print(matchingHighIncome)

matchingHighIncome= matchingHighIncome.sample(n = 5, random_state=  42) # We had over 50+ countries so we "randomly" sampled 5 countries
# with a seed to give us 5 countries chosen for our total 10 list.
print(matchingHighIncome)

# Now we have our 10 countries. 5 low income (matchingLowIncome) + 5 high income (matchingHighIncome)
# Low-income (5): Afghanistan, Ethiopia, Guinea, Malawi, Uganda
# High-income (5): Antigua and Barbuda, Belgium, Norway, Denmark, Seychelles


146
116
102
92


                 country  People_Vaccinated_per_hundred nulls  new_deaths_smoothed_per_million  loss of data due to null (%)  Start Date
7    Antigua and Barbuda                                  484                                0                     83.737024  2021-02-16
19               Belgium                                   29                                0                      2.857143  2020-12-28
157               Norway                                   14                                0                      1.966292  2020-12-02
55               Denmark                                  870                                0                     85.629921  2020-12-18
189           Seychelles                                  717                                0                     87.332521  2021-01-09


In [ ]:
# We are almost done. Now we have to make the adequate csv file for all 10 countries. 

# These are our final countries and now we put it into final_df
final_countries = ["Afghanistan", "Ethiopia", "Guinea", "Malawi", "Uganda", "Antigua and Barbuda", "Belgium", "Norway", "Denmark", "Seychelles"]
final_df = df_cleaned[df_cleaned['location'].isin(final_countries)] 

# now we select the variables we care about only

final_df = final_df[['iso_code', 'location', 'date', 'people_vaccinated_per_hundred', 'new_deaths_smoothed_per_million']]

#print(final_df.shape)
#print(final_df.head())

# I want to make a binary variable column that determines if a country is high income or low income. 
# NOTE: 0 = Low-Income country, 1 = High-Income country

income_lookup = { 
    "Afghanistan": 0, "Ethiopia": 0, "Guinea": 0, "Malawi": 0, "Uganda": 0, "Antigua and Barbuda": 1, "Belgium": 1, "Norway": 1, "Denmark": 1, 
    "Seychelles": 1
}   # This essentially becomes a map for that binary variable I talked about earlier making our work a lot easier

final_df['low/high'] = final_df['location'].map(income_lookup)
print(final_df.shape)
print(final_df.head())
print()
print()
print(final_df.tail())

# One last concern before csv file is made. This is critical. All countries startDates differ making them misleading to visualize on
# one graph that we will attempt to make on our next 02_eda file. I am attempting to create a new column that measures elapsed time since 
# each country's own rollout start (day_since_rollout)

print()
print()
final_df['date'] = pd.to_datetime(final_df['date'])
print(final_df['date'].dtype)

# lets convert and find the time elpased using this for loop
for country in final_countries:
    countryData = final_df[final_df['location'] == country]
    startDate = countryData[countryData['people_vaccinated_per_hundred'].notna()]['date'].min()
    final_df.loc[final_df['location'] == country, 'days_since_rollout'] = (countryData['date'] - startDate).dt.days

# This in summary makes a new clock. A clock where all countries start at the same time. Belgiums data started in December while 
# Unganda's did months later. This clock sets both country's first date to 0. Negative numbers represent data before 0 where 0 represents
# the exact date when the country begins its rollout. This is needed for fair comparison as many countries have widely different timelines
# which might mess our graphs up and becoming misleading when visualzing.

print(final_df[['location', 'date', 'days_since_rollout']].head())
print(final_df[['location', 'date', 'days_since_rollout']].tail())

(16740, 6)
  iso_code     location        date  people_vaccinated_per_hundred  new_deaths_smoothed_per_million  low/high
0      AFG  Afghanistan  2020-01-05                            NaN                              NaN         0
1      AFG  Afghanistan  2020-01-06                            NaN                              NaN         0
2      AFG  Afghanistan  2020-01-07                            NaN                              NaN         0
3      AFG  Afghanistan  2020-01-08                            NaN                              NaN         0
4      AFG  Afghanistan  2020-01-09                            NaN                              NaN         0


       iso_code location        date  people_vaccinated_per_hundred  new_deaths_smoothed_per_million  low/high
398424      UGA   Uganda  2024-07-31                            NaN                              0.0         0
398425      UGA   Uganda  2024-08-01                            NaN                              0.0     

In [ ]:
# finally, make a csv file

final_df.to_csv("cleaned_covid_10countries.csv", index=False)